# Analysis of American Parlimentary Debate Association Rounds Fall 2019 - Spring 2026

## Introduction

...

## Data Curation

The data for this project was sourced at the APDA online forum where, after the conclusion of each tournament, results are posted in the form of PDF Tab Cards. Tab Cards are structured by team where each team has its own table within the PDF. Each table is labelled with the team name, and contains rows that detail the round number, weather the team was in Government or Opposition position, win/loss status, the opponent team name, judge name, the speaks and ranks for each team member, and the total speaks and ranks of the team. For the preliminary data curation, I used pdfplumber to split each PDF into tables in order to extract data for each team. Bounding boxes were used to extract the team names that prefaced each table. Additionally, a reference was necesary to match the team names, which differ by tournament, to the individual debaters in order to creat the columns for the opponent names. It was also necesary to collapse the mirror rows as each match is represented twice, one for the Government teams table and the other in the Opposition team table. Regex patterns were used for cleaning and normalizing the data.  

In [181]:
# imports used for parsing data from PDFs
from pathlib import Path
import re
import logging
import pandas as pd
import pdfplumber

logging.getLogger("pdfminer").setLevel(logging.ERROR)

In [182]:
def get_tab_cards(folder, season, year):
    """
    Creates file paths based on the contents of the data folders. 
    Uses a dictionary to replace file shortenings with the official APDA school name.
    Returns a list for each season's input of the file path, school name, season, and year.
    """
    tab_cards = [] 
    
    for pdf in Path(folder).glob(f"*_{season}{year}_Tab_Card.pdf"):
        school = pdf.name.replace(f"_{season}{year}_Tab_Card.pdf", "")     
        school = {
            "Binghamton": "Binghamton University",
            "BrownAndWesleyan":"Brown + Wesleyan",
            "Chicago":"University of Chicago",
            "ChicagoNortheastern":"University of Chicago + Northeastern",
            "CMU":"Carnegie Mellon",
            "ColumbiaSwarthmore":"Columbia + Swarthmore",
            "CUNYUMD":"CUNY + Maryland",
            "Delaware":"University of Delaware",
            "FranklinAndMarshall": "Franklin and Marshall",
            "Hopkins":"Johns Hopkins",
            "GU":"Georgetown",
            "GW":"George Washington",
            "JHUAU":"Johns Hopkins + American",
            "NU":"Northeastern",
            "NUBC":"Northeastern + Boston College",
            "NYUWashU":"NYU + Washington University",
            "Pitt":"University of Pittsburgh",
            "PittCMU": "University of Pittsburgh + Carnegie Mellon",
            "PrincetonNUBates":"Princeton + Northeastern + Bates",
            "SmithColumbia":"Smith + Columbia",
            "TheCollegeOfNewJersey": "The College of New Jersey",
            "TCNJ": "The College of New Jersey",
            "TempleWesleyan":"Temple + Wesleyan",
            "Tufts2": "Tufts",
            "UMD":"Maryland",
            "UMDBU":"Maryland + Boston University",
            "UMass":"University of Massachusetts",
            "UMass_Amherst":"University of Massachusetts + Amherst",
            "UVA":"University of Virginia",
            "UVAWDS":"University of Virginia + Wellesley",
            "WashU": "Washington University",
            "WestPoint": "West Point",
            "WesleyanTCNJ":"Wesleyan + The College of New Jersey",
            "WilliamAndMary": "William and Mary",
            "William&Mary": "William and Mary",
        }.get(school, school)
        
        tab_cards.append(
            (str(pdf), school, season, year)
        )
    
    return tab_cards

In [183]:
# Initializes tab cards from Fall 2019 - Spring 2026
FALL_2019_TAB_CARDS = get_tab_cards("Fall2019", "Fall", 2019)
SPRING_2020_TAB_CARDS = get_tab_cards("Spring2020", "Spring", 2020)
FALL_2020_TAB_CARDS = get_tab_cards("Fall2020", "Fall", 2020)
SPRING_2021_TAB_CARDS = get_tab_cards("Spring2021", "Spring", 2021)
FALL_2021_TAB_CARDS = get_tab_cards("Fall2021", "Fall", 2021)
SPRING_2022_TAB_CARDS = get_tab_cards("Spring2022", "Spring", 2022)
FALL_2022_TAB_CARDS = get_tab_cards("Fall2022", "Fall", 2022)
SPRING_2023_TAB_CARDS = get_tab_cards("Spring2023", "Spring", 2023)
FALL_2023_TAB_CARDS = get_tab_cards("Fall2023", "Fall", 2023)
SPRING_2024_TAB_CARDS = get_tab_cards("Spring2024", "Spring", 2024)
FALL_2024_TAB_CARDS = get_tab_cards("Fall2024", "Fall", 2024)
SPRING_2025_TAB_CARDS = get_tab_cards("Spring2025", "Spring", 2025)
FALL_2025_TAB_CARDS = get_tab_cards("Fall2025", "Fall", 2025)
SPRING_2026_TAB_CARDS = get_tab_cards("Spring2026", "Spring", 2026)

Several tournaments were excluded for bad formating, lack of tab cards, or lack of permission to the documnets. The following describes those ommited:

In [184]:
# Formatting of the excluded tournaments
from tabulate import tabulate
omitted_tables = [["Fall2019","Fordham, UVA, Columbia","",""],["Spring2020","","GU, Williams",""],
                  ["Fall2020","","Harvard, Swarthmore",""],["Spring2021","Swarthmore","Brandeis, W&M, GU, Yale",""],
                  ["Fall2021","","","Tufts/UMD, GW/Fordham, Brown, Harvard/Penn"],["Spring2022","Princeton, Yale, Rutgers","Brown, Williams","Darthmouth"],
                  ["Fall2022","","Brown, Yale, Williams","Tufts, Harvard, Rutgers"],["Spring2023","Hopkins","Brandeis, Rutgers, Penn, Williams, UChicago","UMass"],
                  ["Fall2023","Hopkins, Swarthmore","Brandeis, Harvard",""],["Spring2024","Amherst","UMass, Brandeis, NYU, Windsor, Tufts",""],
                  ["Fall2024","","Harvard, Binghamton, Tufts, Brown",""],["Spring2025","Rutgers, Amherst, Penn, UVA","","Dartmouth, UT Austin"],
                  ["Fall2025","Drexel","","Brandeis, Bates, American"],["Spring2026","Temple","",""]]
print(tabulate(omitted_tables, headers=["Season","Bad Format", "Lacking Permission", "Missing Tab Card"], tablefmt="grid"))

+------------+-----------------------------+---------------------------------------------+--------------------------------------------+
| Season     | Bad Format                  | Lacking Permission                          | Missing Tab Card                           |
+============+=============================+=============================================+============================================+
| Fall2019   | Fordham, UVA, Columbia      |                                             |                                            |
+------------+-----------------------------+---------------------------------------------+--------------------------------------------+
| Spring2020 |                             | GU, Williams                                |                                            |
+------------+-----------------------------+---------------------------------------------+--------------------------------------------+
| Fall2020   |                             | Har

Below are the regex patterns used for normalization and sanitization as well as the functions used to parse and clean the df:

In [185]:
# pre-compile regex patterns
MULTISPACE_RX = re.compile(r"\s+")
STATUS_PAREN_RX = re.compile(r"\s*\(([NV])\)\s*$")
CLEAN_TRAILING_RX = re.compile(r"\s+$")
TEAM_PREFIX_RX = re.compile(r"Team:\s*(.+)")
EMOJI_RX = re.compile(
    r"[\U00010000-\U0010FFFF"  
    r"\u2300-\u23FF"           
    r"\u2600-\u26FF"           
    r"\u2700-\u27BF"          
    r"\u2B00-\u2BFF"           
    r"\uFE0F"                  
    r"\u200D]"                 
)

NUMBER_RX = re.compile(r"\d")
KEYCAP_RX = re.compile(r"[\u20E3]")
TM_RX = re.compile(r"™")

Below are the functions used in parsing and cleaning:

In [186]:
def normalize_cell(name):
    """
    Fills NaN with an empty string, removes emojis, and removes spaces
    """
    if name is None:
        return ""

    name = str(name).title()
    name = EMOJI_RX.sub("", name)
    name = KEYCAP_RX.sub("", name)
    name = TM_RX.sub("", name)

    return MULTISPACE_RX.sub(" ", name).strip()

def normalize_names(name):
    """
    Removes numbers
    """
    if name is None:
        return ""

    name = str(name)
    name = NUMBER_RX.sub("", name)

    return normalize_cell(name)
    
def split_name_and_status(raw_name):
    """
    Return (clean_name, status) extracted from a speaker header.
    """
    if not raw_name:
        return "", None

    name = normalize_names(raw_name)
    match = STATUS_PAREN_RX.search(name)

    if match:
        status = "Novice" if match.group(1) == "N" else "Varsity"
        return STATUS_PAREN_RX.sub("", name).strip(), status

    return name, None

def split_speaks_and_ranks(df, score_col, prefix):
    """
    Split a score column into separate speaks and rank columns.
    """
    if score_col not in df.columns:
        return df

    cleaned_series = df[score_col].astype(str).str.replace(r"[\(\)\s]", "", regex=True)
    split_data = cleaned_series.str.split(",", expand=True)

    if split_data.shape[1] < 2:
        split_data = pd.DataFrame(index=df.index, columns=[0, 1])

    df[f"{prefix} Speaks"] = pd.to_numeric(split_data[0], errors="coerce")
    df[f"{prefix} Rank"] = pd.to_numeric(split_data[1], errors="coerce").astype("Int64") # Capital I allows NaN integers

    return df.drop(columns=[score_col])

def add_speaker_and_opponent_names(all_df, speaker_df):
    """
    Matches teams and opponents using normalized team keys, then splits
    speaker scores into speaks and rank columns.
    """
    speaker_df = speaker_df.copy()
    speaker_df["_key"] = speaker_df["Team"].map(normalize_names)
    speaker_df = speaker_df.drop_duplicates(subset="_key", keep="first")

    lookup = speaker_df.set_index("_key")[
        ["Speaker One Name", "Speaker One Status", "Speaker Two Name", "Speaker Two Status"]
    ]

    opponent_lookup = lookup.rename(columns={
        "Speaker One Name": "Opponent One Name",
        "Speaker One Status": "Opponent One Status",
        "Speaker Two Name": "Opponent Two Name",
        "Speaker Two Status": "Opponent Two Status"
    })
    
    df = all_df.copy()
    df["Judge"] = df["Judge"].map(normalize_names)
    df["_team_key"] = df["Team"].map(normalize_names)
    df["_opponent_key"] = df["Opponent"].map(normalize_names)
  
    df = df.merge(lookup, left_on="_team_key", right_index=True, how="left")
    df = df.merge(opponent_lookup, left_on="_opponent_key", right_index=True, how="left")


    score_lookup = all_df.copy()
    score_lookup["_opponent_key"] = score_lookup["Team"].map(normalize_names)
    
    score_lookup = (
        score_lookup[
            ["_opponent_key", "Round", "Speaker One Score", "Speaker Two Score"]
        ]
        .drop_duplicates(["_opponent_key", "Round"])
        .rename(columns={
            "Speaker One Score": "Opponent Speaker One Score",
            "Speaker Two Score": "Opponent Speaker Two Score"
        })
    )
    df = df.merge(score_lookup, on=["_opponent_key", "Round"], how="left")
    df = df.drop(columns=["Team", "Opponent", "_team_key", "_opponent_key"])

    df = split_speaks_and_ranks(df, "Speaker One Score", "Speaker One")
    df = split_speaks_and_ranks(df, "Speaker Two Score", "Speaker Two")
    df = split_speaks_and_ranks(df, "Opponent Speaker One Score", "Opponent One")
    df = split_speaks_and_ranks(df, "Opponent Speaker Two Score", "Opponent Two")
    
    ordered_cols = [c for c in [
        "Tournament", "Season", "Year", "Round", "G/O", "W/L",
        "Speaker One Name", "Speaker One Status", "Speaker Two Name", "Speaker Two Status",
        "Opponent One Name", "Opponent One Status", "Opponent Two Name", "Opponent Two Status",
        "Judge", "Speaker One Speaks", "Speaker One Rank", "Speaker Two Speaks", "Speaker Two Rank",
        "Opponent One Speaks", "Opponent One Rank", "Opponent Two Speaks", "Opponent Two Rank", "Total",
    ] if c in df.columns]
    
    return df[ordered_cols]

def parse_tab_cards(card):
    """
    Parses the individual tournament tab card
    """
    all_rows, speaker_lookup = [], []
    seen_teams = set()
    current_team, current_speakers, pending_label = None, None, None
    
    with pdfplumber.open(card) as pdf:
        for page in pdf.pages:
            team_labels = [] # dictionaries for team names + locations
            for line in page.extract_text_lines():
                match = TEAM_PREFIX_RX.search(line["text"])
                if match:
                    team_labels.append({"name": match.group(1).strip(), "top": line["top"]})
 
            tables = page.find_tables()
 
            for table in tables:
                data = table.extract()
                if not data or len(data) < 1:
                    continue
    
                first_cell = normalize_cell(data[0][0]) if data[0] else ""
                has_header = (first_cell in ("R", "")) # stores whether the first cell is the beginning of the table 
 
                if has_header: # if a new table, store 
                    header = [normalize_cell(c) for c in data[0]]
                    body = data[1:]
                    speaker1_header = header[5] if len(header) > 5 else ""
                    speaker2_header = header[6] if len(header) > 6 else ""
                else: # if a continuation of the last table
                    body = data
                    speaker1_header, speaker2_header = current_speakers if current_speakers else ("", "")

                # finds the team name stored in team_labels that is above current table
                table_top = table.bbox[1]
                labels_above = [t for t in team_labels if t["top"] <= table_top]
 
                if labels_above:
                    team_name = normalize_cell(
                        max(labels_above, key=lambda t: t["top"])["name"]
                    )
                    current_team = team_name
                    current_speakers = (speaker1_header, speaker2_header)
                    pending_label = None
                elif has_header and current_speakers and (speaker1_header, speaker2_header) == current_speakers: # handles table breaking to next page with label headings
                    team_name = current_team
                elif not has_header and current_team is not None: # handles continuation table w/o header
                    team_name = current_team
                elif pending_label is not None: # detected team name before
                    team_name = pending_label
                    current_team = team_name
                    current_speakers = (speaker1_header, speaker2_header)
                    pending_label = None
                else:
                    team_name = current_team
                    current_speakers = (speaker1_header, speaker2_header)

                if team_name not in seen_teams:
                    seen_teams.add(team_name) # set for O(1) lookup
                    name1, status1 = split_name_and_status(speaker1_header)
                    name2, status2 = split_name_and_status(speaker2_header)
                    speaker_lookup.append({
                        "Team": team_name,
                        "Speaker One Name": name1, "Speaker One Status": status1,
                        "Speaker Two Name": name2, "Speaker Two Status": status2,
                    })
 
                for row in body:
                    row = [normalize_cell(c) for c in row]
                    if not row or row[0].lower().startswith("tournament totals"):
                        continue  
 
                    row = (row + [""] * 8)[:8]
                    round_no, g_o, w_l, opponent, judge, sp1, sp2, total = row
 
                    if not round_no:
                        if not any([g_o, w_l, opponent, judge, sp1, sp2]):
                            continue
                        round_no = "UNKNOWN (split across page break)"
 
                    all_rows.append({
                        "Team": team_name, "Round": round_no, "G/O": g_o, "W/L": w_l,
                        "Opponent": opponent, "Judge": judge, 
                        "Speaker One Score": sp1, "Speaker Two Score": sp2, "Total": total,
                    })
 
            if team_labels:
                last_label = max(team_labels, key=lambda t: t["top"])
                table_tops = [t.bbox[1] for t in tables]
                if not any(last_label["top"] <= top for top in table_tops):
                    pending_label = normalize_cell(last_label["name"])
 
    return pd.DataFrame(all_rows), pd.DataFrame(speaker_lookup)

def process_all_cards(tab_card):
    """
    Calls parse then adds Tournament, Season, and Year column while dropping Total. Combines all semester rows + speakers into one row
    """
    semester_rounds_df = [] 
    semester_speakers_df = [] 
 
    for pdf_path, tournament_name, season, year in tab_card:
        tournament_rounds_df, tournament_speaker_df = parse_tab_cards(pdf_path)
        
        if tournament_rounds_df.empty:
            continue
        tournament_rounds_df = tournament_rounds_df.drop(columns=["Total"], errors="ignore")
        tournament_rounds_df.insert(0, "Tournament", tournament_name)
        tournament_rounds_df.insert(1, "Season", season)
        tournament_rounds_df.insert(2, "Year", year)

        tournament_rounds_df = add_speaker_and_opponent_names(tournament_rounds_df, tournament_speaker_df)
        semester_rounds_df.append(tournament_rounds_df)
 
        tournament_speaker_df.insert(0, "Tournament", tournament_name)
        tournament_speaker_df.insert(1, "Season", season)
        tournament_speaker_df.insert(2, "Year", year)
        
        semester_speakers_df.append(tournament_speaker_df)
 
    return pd.concat(semester_rounds_df, ignore_index=True), pd.concat(semester_speakers_df, ignore_index=True)

Now we can parse the rounds

In [187]:
# Assigns a label for each semester of tab cards
semester_tab_paths =  [(FALL_2019_TAB_CARDS, "Fall_2019.csv"), 
                       (SPRING_2020_TAB_CARDS, "Spring_2020.csv"), (FALL_2020_TAB_CARDS, "Fall_2020.csv"),
                       (SPRING_2021_TAB_CARDS, "Spring_2021.csv"), (FALL_2021_TAB_CARDS, "Fall_2021.csv"),
                       (SPRING_2022_TAB_CARDS, "Spring_2022.csv"), (FALL_2022_TAB_CARDS, "Fall_2022.csv"),
                       (SPRING_2023_TAB_CARDS, "Spring_2023.csv"), (FALL_2023_TAB_CARDS, "Fall_2023.csv"),
                       (SPRING_2024_TAB_CARDS, "Spring_2024.csv"), (FALL_2024_TAB_CARDS, "Fall_2024.csv"),
                       (SPRING_2025_TAB_CARDS, "Spring_2025.csv"), (FALL_2025_TAB_CARDS, "Fall_2025.csv"),
                       (SPRING_2026_TAB_CARDS, "Spring_2026.csv")
                      ]
# Creates a df of the rounds and a speaker_df with all speakers seen
dfs = []
speakers = []
for (semester_card, label) in semester_tab_paths:
    df, speaker_df = process_all_cards(semester_card)

    # Cleans df by removing BYE rows or rows where speaker names are missing or ranks/speaks are zero
    df = df[
        ~df["G/O"].fillna("").str.strip().isin(["", "BYE"])
        & df["Speaker One Name"].fillna("").str.strip().ne("")
        & df["Speaker Two Name"].fillna("").str.strip().ne("")
        & df["Opponent One Name"].fillna("").str.strip().ne("")
        & df["Opponent Two Name"].fillna("").str.strip().ne("")
        & (df["Speaker One Rank"] != 0)
        & (df["Speaker Two Rank"] != 0)
        & (df["Opponent One Rank"] != 0)
        & (df["Opponent Two Rank"] != 0)
        & (df["Speaker One Speaks"] != 0.0)
        & (df["Speaker Two Speaks"] != 0.0)
        & (df["Opponent One Speaks"] != 0.0)
        & (df["Opponent Two Speaks"] != 0.0)
    ]
    df["Judge"] = df["Judge"].map(normalize_team_name)
    df['Judge'] = (
        df['Judge']
        .str.split(' - ').str[0]
        .str.replace(r'\s*\(V\)\s*', '', regex=True)
        .str.strip()
    )

    df.to_csv(label, index=False)
    dfs.append(df)
    speakers.append(speaker_df)

Combine and save the rounds

In [192]:
rounds_dfs = pd.concat(dfs, ignore_index=True)
rounds_dfs.to_csv("Rounds.csv", index=False)

Now we need a list of debaters and the schools they are from 

In [231]:
from selenium import webdriver
from bs4 import BeautifulSoup
from io import StringIO
import random

In [232]:
school_ids = ["18"]
years = ["2019", "2020", "2021", "2022", "2023", "2024", "2025", "2026"]

In [ ]:
driver = webdriver.Chrome()
all_debaters = []

for i in school_ids:
    for y in years:
        url = "https://results.apda.online/core/schools/" + i + "?season=" + y
        driver.get(url)
        input("Complete the CAPTCHA, then press Enter...")
        
        soup = BeautifulSoup(driver.page_source, "html.parser")

        target_table = None

        for table in soup.find_all("table"):
            headers = [th.get_text(strip=True) for th in table.find_all("th")]
            if headers == ["ID", "Name", "Year on Team"]:
                target_table = table
                break

        if target_table:
            school_debaters = pd.read_html(StringIO(str(target_table)))[0]
            school_debaters["School ID"] = i
            school_debaters["Season"] = y
            all_debaters.append(school_debaters)
        else:
            print(f"No roster found: school {i}, season {y}")

driver.quit()

all_debaters_df = pd.concat(all_debaters, ignore_index=True)
all_debaters_df.to_csv("all_debaters.csv", index=False)

Complete the CAPTCHA, then press Enter... 
Complete the CAPTCHA, then press Enter... 


No roster found: school 18, season 2020


Complete the CAPTCHA, then press Enter... 


No roster found: school 18, season 2021


In [758]:
valid_name_rows = set(sorted_valid_df["Debater"])
clean_rounds_over_time = clean_rounds_over_time[
    #clean_rounds_over_time["Speaker One Name"].isin(valid_names)
    #& clean_rounds_over_time["Speaker Two Name"].isin(valid_names)
    clean_rounds_over_time["Judge"].isin(valid_names)
]
clean_rounds_over_time.to_csv("All_Rounds.csv", index=False)

## Exploratory Analysis

In [760]:
clean_rounds_over_time = clean_rounds_over_time.rename(columns={"Judge": "Debater"})
judge_school_df = pd.merge(clean_rounds_over_time, merged_df, on='Debater', how='inner')
judge_school_df = judge_school_df.rename(columns={"Debater": "Judge", "University": "Judge School", "Starting Year": "Judge Start"})
judge_school_df = judge_school_df.drop_duplicates()

judge_school_df.to_csv("Rounds_With_Judge_School.csv", index=False)

In [762]:
speaks_one = judge_school_df.groupby('Judge School')['Speaker One Speaks'].mean()
speaks_two = judge_school_df.groupby('Judge School')['Speaker Two Speaks'].mean()
opp_speaks_one = judge_school_df.groupby('Judge School')['Opponent One Speaks'].mean()
opp_speaks_two = judge_school_df.groupby('Judge School')['Opponent Two Speaks'].mean()
count = judge_school_df.groupby('Judge School')['Opponent Two Speaks'].count()
total_speaks_avg = (speaks_one + speaks_two + opp_speaks_one + opp_speaks_two) / 4

### Speaks

## Insights and Conclusions